# Мой проект: разовый план без погоды
До финального теста: сравнить frozen_week, linear_interactions, trend_hgb по RMSE апреля–июня. Фиксированный бюджет 200 HGB итераций, depth3, lr.05. Гипотеза: тренд уменьшит систематическое занижение. Единицы — тыс. ₽ на кофейню в день.


In [1]:
import os,sys,runpy,json
from pathlib import Path
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[name]='1'
import numpy as np,pandas as pd,sklearn
RANDOM_STATE=11
print('Python',sys.version.split()[0],'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
from IPython.display import display


Python 3.12.14 numpy 2.4.6 pandas 3.0.6 sklearn 1.8.0


## Цель
Снизить RMSE относительно замороженного недельного ориентира для разового плана выпуска1июля. Кандидат выбирается по валидации, результаты финального периода сообщаются без смены победителя. Заказчик получает plan.csv и воспроизводимый пересчёт.


## Данные и предобработка
Учебный daily_sales.csv, 2024–2025, одна кофейня в один день. Фактическая погода исключена из всех кандидатов. Календарь/акции считаются утверждёнными к выпуску. Проверяю пропуски и повторные ключи; строки не удаляю без основания.


In [2]:
data=pd.read_csv('data/daily_sales.csv',parse_dates=['date'])
print(data.shape, data.date.min(), data.date.max())
print('missing',data.isna().sum().to_dict())
print('duplicate shop/date',data.duplicated(['shop_id','date']).sum())


(4386, 9) 2024-01-01 00:00:00 2025-12-31 00:00:00
missing {'date': 0, 'shop_id': 0, 'city': 0, 'format': 0, 'temp': 0, 'rain': 0, 'holiday': 0, 'promo': 0, 'revenue_k': 0}
duplicate shop/date 0


**Вывод по данным.** Учебный набор содержит6кофеен; качество и уникальность ключей проверяются предыдущей ячейкой. Календарная шкала фиксирована от1января2024, категории обучаются на истории и сохраняются в артефакте.


## Анализ и модели
Линейные взаимодействия фиксированы заранее: бизнес-центр/ТЦ × нерабочий день. Тренд — линейная регрессия log(revenue_k) на day_n, HGB на нормированной выручке без day_n. Факты будущих месяцев не участвуют в признаках или переобучении.


In [3]:
sys.argv=["run.py"]
_ = runpy.run_path('run.py',run_name='__main__')


chosen by validation: trend_hgb
                     val_rmse_k  selection_fit_s  final_fit_s  predict_ms  test_rmse_k  test_mae_k    bias_k
model                                                                                                       
trend_hgb              7.688059         0.056272     0.062042    4.598458     8.672296    6.335018 -0.537930
linear_interactions    8.239695         0.002620     0.002753    2.380666     9.762934    7.109239  0.446219
frozen_week           13.527024         0.003657     0.003571    1.709833    14.480730   10.494112  1.412953
reloaded model and single-row agreement: OK


In [4]:
display(pd.read_csv('metrics.csv'))
display(pd.read_csv('plan.csv').head())


,model,val_rmse_k,selection_fit_s,final_fit_s,predict_ms,test_rmse_k,test_mae_k,bias_k
0,trend_hgb,7.688059,0.056272,0.062042,4.598458,8.672296,6.335018,-0.537930
1,linear_interactions,8.239695,0.002620,0.002753,2.380666,9.762934,7.109239,0.446219
2,frozen_week,13.527024,0.003657,0.003571,1.709833,14.480730,10.494112,1.412953


,date,shop_id,prediction_k
0,2025-07-01,KZN-1,85.057383
1,2025-07-01,KZN-2,88.013883
2,2025-07-01,PRM-1,71.103605
3,2025-07-01,PRM-2,54.276859
4,2025-07-01,TMN-1,56.191121


**Вывод по анализу.** Валидационный выбор trend_hgb дал RMSE7.688 тыс. ₽, лучше линейных взаимодействий8.240 и ориентира13.527. На финальном периоде RMSE8.672 ниже ориентира14.481; разница5.808 тыс. ₽ полезна для точности плана, но не равна экономии ресурсов. Проверка reload и одиночной строки прошла.


## Выводы и рекомендации
На валидации выбран trend_hgb: RMSE 7.688 тыс. ₽ против13.527 у замороженного недельного ориентира. На финальном полугодии RMSE8.672, MAE6.335 тыс. ₽, средняя ошибка факт−прогноз −0.538 тыс. ₽. Ориентир дал RMSE14.481, линейная модель9.763. Снижение RMSE относительно ориентира5.808 тыс. ₽ (около40%).
Финальный fit занял0.062с, пакетный predict4.59мс в этом запуске; реальные интервалы сохраняются в metrics.csv. Рекомендую пилот разового плана с контролем ошибок по кофейням; эти результаты не измеряют экономию смен или молока. Погода исключена, утверждение календаря акций предположено. Неопределённость и устойчивость на новом бизнесе ещё не проверены.
Загруженная модель дала тот же пакет и ту же одиночную строку. README и request.csv позволяют коллеге пересчитать прогноз без целевых фактов.
